# Human-in-the-Loop RAG Evaluation [Step 3 - Grading Interface and Aggregation]

> **MLCourse - Agentic AI - RAG Evaluation**

Automated metrics are useful but cannot fully capture answer quality.
Human evaluation remains the gold standard for:
- Nuanced correctness judgments
- Detecting subtle hallucinations
- Assessing answer helpfulness and tone
- Validating automated metric scores

This notebook builds a human evaluation system with:
1. A grading interface that presents RAG outputs for human review
2. Multiple grading dimensions (correctness, relevance, completeness, safety)
3. Score aggregation across multiple graders
4. Inter-annotator agreement analysis

In [1]:
# ## 1. Setup and Environment

In [2]:
import os
import json
import uuid
import datetime
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

print("[setup] Human evaluation notebook initialized")

[setup] Human evaluation notebook initialized


In [3]:
# ## 2. Initialize the LLM

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("[llm] ChatOllama ready:", llm.model)

[llm] ChatOllama ready: llama3.1:8b


In [5]:
# ## 3. Build the RAG Pipeline (source of outputs to evaluate)

In [6]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

ALICE_PATH = Path(r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt")
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")[:25_000]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_text(raw_text)
documents = [
    Document(page_content=c, metadata={"chunk_id": i})
    for i, c in enumerate(chunks)
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(f"[Chunk {d.metadata.get('chunk_id', i)}] {d.page_content}"
                       for i, d in enumerate(docs))

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about Alice in Wonderland using the provided context. "
     "Cite chunk numbers like [Chunk 3] when referencing information."),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("[rag] RAG pipeline ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[rag] RAG pipeline ready


In [7]:
# ## 4. Define Grading Rubrics
# We define clear rubrics for each grading dimension.

In [8]:
rubrics = {
    "correctness": {
        "description": "Is the answer factually correct based on the source text?",
        "scale": {
            1: "Completely wrong or contradicts the source",
            2: "Mostly wrong with some correct elements",
            3: "Partially correct, missing key facts",
            4: "Mostly correct with minor inaccuracies",
            5: "Completely correct and accurate",
        },
    },
    "relevance": {
        "description": "Does the answer address the question asked?",
        "scale": {
            1: "Completely irrelevant to the question",
            2: "Mostly irrelevant, tangentially related",
            3: "Partially relevant, misses the main point",
            4: "Relevant with minor off-topic elements",
            5: "Directly and fully addresses the question",
        },
    },
    "completeness": {
        "description": "Does the answer cover all important aspects of the question?",
        "scale": {
            1: "Extremely incomplete, missing most information",
            2: "Largely incomplete, covers only one aspect",
            3: "Partially complete, covers main points but misses details",
            4: "Mostly complete, minor details missing",
            5: "Fully comprehensive, covers all aspects",
        },
    },
    "safety": {
        "description": "Does the answer contain any harmful, biased, or inappropriate content?",
        "scale": {
            1: "Contains harmful or inappropriate content",
            2: "Slightly inappropriate or biased",
            3: "Neutral, no obvious issues",
            4: "Safe and considerate",
            5: "Exemplary safety, helpful framing",
        },
    },
}

print("[rubric] Defined 4 grading dimensions with 5-point scales:")
for dim, info in rubrics.items():
    print(f"  - {dim}: {info['description']}")

[rubric] Defined 4 grading dimensions with 5-point scales:
  - correctness: Is the answer factually correct based on the source text?
  - relevance: Does the answer address the question asked?
  - completeness: Does the answer cover all important aspects of the question?
  - safety: Does the answer contain any harmful, biased, or inappropriate content?


In [9]:
# ## 5. Create Evaluation Dataset
# Generate RAG outputs for a set of questions and package them for human review.

In [10]:
questions = [
    "Why did Alice follow the White Rabbit?",
    "What did the Caterpillar ask Alice?",
    "How did the Mad Hatter describe the tea situation?",
    "What did the Cheshire Cat say about madness?",
    "What happened during the trial at the end?",
]

eval_items = []
for q in questions:
    answer = rag_chain.invoke(q)
    docs = retriever.invoke(q)
    contexts = [d.page_content for d in docs]

    item_id = str(uuid.uuid4())[:8]
    eval_items.append({
        "id": item_id,
        "question": q,
        "answer": answer,
        "contexts": contexts,
        "graders": {},
        "created_at": datetime.datetime.now().isoformat(),
    })

print(f"[dataset] Created {len(eval_items)} items for human evaluation")
for item in eval_items:
    print(f"  [{item['id']}] {item['question'][:50]}...")

[dataset] Created 5 items for human evaluation
  [b769b2e4] Why did Alice follow the White Rabbit?...
  [c81f0823] What did the Caterpillar ask Alice?...
  [3b407e90] How did the Mad Hatter describe the tea situation?...
  [8ba4f184] What did the Cheshire Cat say about madness?...
  [09f13f3f] What happened during the trial at the end?...


In [11]:
# ## 6. Build the Grading Interface
# This class manages the human evaluation workflow: presenting items,
# collecting grades, and tracking progress.

In [12]:
class HumanGradingInterface:
    """Manages human evaluation of RAG outputs."""

    def __init__(self, eval_items, rubrics):
        self.items = eval_items
        self.rubrics = rubrics
        self.grades = {}  # item_id -> grader_id -> {dimension: score}
        self.grader_ids = set()

    def display_item(self, item_id):
        """Display an evaluation item for grading."""
        item = next((i for i in self.items if i["id"] == item_id), None)
        if not item:
            print(f"Item {item_id} not found")
            return

        print("=" * 60)
        print(f"ITEM: {item_id}")
        print(f"QUESTION: {item['question']}")
        print(f"\nRAG ANSWER:")
        print(f"  {item['answer']}")
        print(f"\nRETRIEVED CONTEXT ({len(item['contexts'])} chunks):")
        for i, ctx in enumerate(item["contexts"]):
            print(f"  [Chunk {i}] {ctx[:80]}...")
        print("=" * 60)

    def grade_item(self, item_id, grader_id, scores):
        """Record grades for an item from a specific grader.

        Args:
            item_id: The evaluation item ID
            grader_id: Unique identifier for the grader
            scores: Dict of {dimension: score} where score is 1-5
        """
        if item_id not in self.grades:
            self.grades[item_id] = {}

        # Validate scores
        validated = {}
        for dim, score in scores.items():
            if dim in self.rubrics:
                validated[dim] = max(1, min(5, int(score)))
            else:
                print(f"  Warning: unknown dimension '{dim}', skipping")

        self.grades[item_id][grader_id] = validated
        self.grader_ids.add(grader_id)
        print(f"  Recorded grades for item {item_id} by grader {grader_id}")

    def get_item_grades(self, item_id):
        """Get all grades for a specific item."""
        return self.grades.get(item_id, {})

    def summary(self):
        """Print a summary of grading progress."""
        total = len(self.items)
        graded = sum(1 for item in self.items if item["id"] in self.grades)
        print(f"\nGrading Progress: {graded}/{total} items graded")
        print(f"Active graders: {len(self.grader_ids)}")
        for gid in self.grader_ids:
            count = sum(1 for item in self.items
                        if item["id"] in self.grades
                        and gid in self.grades[item["id"]])
            print(f"  Grader {gid}: {count} items graded")

In [13]:
# Initialize the grading interface.

In [14]:
grader = HumanGradingInterface(eval_items, rubrics)
print("[interface] Grading interface ready")

[interface] Grading interface ready


In [15]:
# ## 7. Simulate Multiple Human Graders
# In practice, multiple humans grade the same items to measure agreement.
# Here we simulate 3 graders with different grading tendencies.

In [16]:
import random
random.seed(42)

def simulate_grader(eval_items, rubrics, grader_id, bias=0, noise=1):
    """Simulate a human grader with optional bias and noise.

    Args:
        bias: Shift all scores by this amount (positive = lenient)
        noise: Add random noise to scores
    """
    interface = HumanGradingInterface(eval_items, rubrics)

    for item in eval_items:
        scores = {}
        for dim in rubrics:
            base = random.randint(3, 5)  # base quality
            noisy = int(base + bias + random.gauss(0, noise))
            scores[dim] = max(1, min(5, noisy))
        interface.grade_item(item["id"], grader_id, scores)

    return interface

# Simulate three graders with different tendencies.
grader_strict = simulate_grader(eval_items, rubrics, "grader_A", bias=-1, noise=0.5)
grader_neutral = simulate_grader(eval_items, rubrics, "grader_B", bias=0, noise=0.3)
grader_lenient = simulate_grader(eval_items, rubrics, "grader_C", bias=1, noise=0.5)

print("[sim] Simulated 3 graders with different tendencies:")
print("  grader_A: strict (bias=-1)")
print("  grader_B: neutral (bias=0)")
print("  grader_C: lenient (bias=+1)")

  Recorded grades for item b769b2e4 by grader grader_A
  Recorded grades for item c81f0823 by grader grader_A
  Recorded grades for item 3b407e90 by grader grader_A
  Recorded grades for item 8ba4f184 by grader grader_A
  Recorded grades for item 09f13f3f by grader grader_A
  Recorded grades for item b769b2e4 by grader grader_B
  Recorded grades for item c81f0823 by grader grader_B
  Recorded grades for item 3b407e90 by grader grader_B
  Recorded grades for item 8ba4f184 by grader grader_B
  Recorded grades for item 09f13f3f by grader grader_B
  Recorded grades for item b769b2e4 by grader grader_C
  Recorded grades for item c81f0823 by grader grader_C
  Recorded grades for item 3b407e90 by grader grader_C
  Recorded grades for item 8ba4f184 by grader grader_C
  Recorded grades for item 09f13f3f by grader grader_C
[sim] Simulated 3 graders with different tendencies:
  grader_A: strict (bias=-1)
  grader_B: neutral (bias=0)
  grader_C: lenient (bias=+1)


In [17]:
# ## 8. Aggregate Grades Across Graders
# We combine grades from multiple graders into a final score per item.

In [18]:
def aggregate_grades(items, grade_data):
    """Aggregate grades across all graders for each item.

    Returns per-item averages and per-dimension averages.
    """
    all_grades = grade_data.get_all_grades() if hasattr(grade_data, "get_all_grades") else grade_data
    dimensions = list(rubrics.keys())

    item_averages = {}
    dimension_totals = {dim: [] for dim in dimensions}

    for item in items:
        item_id = item["id"]
        if item_id not in all_grades:
            continue

        grades_for_item = all_grades[item_id]
        dim_avgs = {}

        for dim in dimensions:
            scores = [g[dim] for g in grades_for_item.values() if dim in g]
            if scores:
                avg = sum(scores) / len(scores)
                dim_avgs[dim] = avg
                dimension_totals[dim].append(avg)

        overall = sum(dim_avgs.values()) / len(dim_avgs) if dim_avgs else 0
        item_averages[item_id] = {
            "dimensions": dim_avgs,
            "overall": overall,
            "num_graders": len(grades_for_item),
        }

    # Per-dimension averages across all items
    dimension_averages = {}
    for dim in dimensions:
        vals = dimension_totals[dim]
        dimension_averages[dim] = sum(vals) / len(vals) if vals else 0

    return item_averages, dimension_averages

In [19]:
# Combine grades from all three simulated graders.

In [20]:
combined_grades = {}
for item in eval_items:
    item_id = item["id"]
    combined_grades[item_id] = {}
    for gid, g in [("grader_A", grader_strict),
                    ("grader_B", grader_neutral),
                    ("grader_C", grader_lenient)]:
        if item_id in g.grades and gid in g.grades[item_id]:
            combined_grades[item_id][gid] = g.grades[item_id][gid]

item_avgs, dim_avgs = aggregate_grades(eval_items, combined_grades)

print("Aggregated Scores Per Item")
print("=" * 60)
for item in eval_items:
    item_id = item["id"]
    if item_id in item_avgs:
        info = item_avgs[item_id]
        print(f"\n  [{item_id}] {item['question'][:50]}...")
        for dim, score in info["dimensions"].items():
            print(f"    {dim:15s}: {score:.2f}")
        print(f"    {'OVERALL':15s}: {info['overall']:.2f}")

print("\nDimension Averages Across All Items")
print("-" * 40)
for dim, avg in dim_avgs.items():
    bar = "#" * int(avg * 4)
    print(f"  {dim:15s}: {avg:.2f} {bar}")

Aggregated Scores Per Item

  [b769b2e4] Why did Alice follow the White Rabbit?...
    correctness    : 3.67
    relevance      : 3.33
    completeness   : 3.67
    safety         : 4.00
    OVERALL        : 3.67

  [c81f0823] What did the Caterpillar ask Alice?...
    correctness    : 4.67
    relevance      : 4.00
    completeness   : 3.33
    safety         : 2.67
    OVERALL        : 3.67

  [3b407e90] How did the Mad Hatter describe the tea situation?...
    correctness    : 3.67
    relevance      : 4.33
    completeness   : 3.33
    safety         : 3.00
    OVERALL        : 3.58

  [8ba4f184] What did the Cheshire Cat say about madness?...
    correctness    : 3.00
    relevance      : 2.67
    completeness   : 3.67
    safety         : 3.00
    OVERALL        : 3.08

  [09f13f3f] What happened during the trial at the end?...
    correctness    : 3.33
    relevance      : 4.67
    completeness   : 3.33
    safety         : 2.67
    OVERALL        : 3.50

Dimension Averages Acro

In [21]:
# ## 9. Inter-Annotator Agreement (IAA)
# Cohen's Kappa and Fleiss' Kappa measure how much graders agree
# beyond chance. Higher agreement means more reliable evaluation.

In [22]:
def compute_pairwise_agreement(grades1, grades2, dimensions):
    """Compute percent agreement between two graders."""
    agreements = 0
    total = 0
    for dim in dimensions:
        scores1 = [g[dim] for g in grades1.values() if dim in g]
        scores2 = [g[dim] for g in grades2.values() if dim in g]
        for s1, s2 in zip(scores1, scores2):
            total += 1
            if s1 == s2:
                agreements += 1
    return agreements / max(total, 1)


def compute_kappa(grades1, grades2, dimensions, num_categories=5):
    """Compute Cohen's Kappa for two graders."""
    agreements = 0
    total = 0
    cat_counts1 = [0] * num_categories
    cat_counts2 = [0] * num_categories

    for dim in dimensions:
        scores1 = [g[dim] for g in grades1.values() if dim in g]
        scores2 = [g[dim] for g in grades2.values() if dim in g]
        for s1, s2 in zip(scores1, scores2):
            total += 1
            if s1 == s2:
                agreements += 1
            cat_counts1[s1 - 1] += 1
            cat_counts2[s2 - 1] += 1

    observed = agreements / max(total, 1)
    expected = sum(c1 * c2 for c1, c2 in zip(cat_counts1, cat_counts2))
    expected = expected / max(total * total, 1)

    if expected == 1.0:
        return 1.0
    kappa = (observed - expected) / (1.0 - expected)
    return kappa

In [23]:
# Compute agreement between all pairs of graders.

In [24]:
dimensions = list(rubrics.keys())

pairs = [
    ("grader_A", "grader_B", grader_strict, grader_neutral),
    ("grader_A", "grader_C", grader_strict, grader_lenient),
    ("grader_B", "grader_C", grader_neutral, grader_lenient),
]

print("Inter-Annotator Agreement")
print("=" * 50)

all_kappas = []
for name1, name2, g1, g2 in pairs:
    # Align grades for same items
    aligned1 = {}
    aligned2 = {}
    for item in eval_items:
        iid = item["id"]
        if iid in g1.grades and iid in g2.grades:
            aligned1[iid] = g1.grades[iid]
            aligned2[iid] = g2.grades[iid]

    agree = compute_pairwise_agreement(aligned1, aligned2, dimensions)
    kappa = compute_kappa(aligned1, aligned2, dimensions)

    all_kappas.append(kappa)

    interpretation = "Poor"
    if kappa > 0.8:
        interpretation = "Almost Perfect"
    elif kappa > 0.6:
        interpretation = "Substantial"
    elif kappa > 0.4:
        interpretation = "Moderate"
    elif kappa > 0.2:
        interpretation = "Fair"

    print(f"\n  {name1} vs {name2}:")
    print(f"    Percent Agreement: {agree:.2%}")
    print(f"    Cohen's Kappa:     {kappa:.4f}")
    print(f"    Interpretation:    {interpretation}")

avg_kappa = sum(all_kappas) / len(all_kappas)
print(f"\n  Average Kappa: {avg_kappa:.4f}")

Inter-Annotator Agreement

  grader_A vs grader_B:
    Percent Agreement: 0.00%
    Cohen's Kappa:     0.0000
    Interpretation:    Poor

  grader_A vs grader_C:
    Percent Agreement: 0.00%
    Cohen's Kappa:     0.0000
    Interpretation:    Poor

  grader_B vs grader_C:
    Percent Agreement: 0.00%
    Cohen's Kappa:     0.0000
    Interpretation:    Poor

  Average Kappa: 0.0000


In [25]:
# ## 10. Generate Human Evaluation Report

In [26]:
def generate_report(items, item_avgs, dim_avgs, kappa_avg):
    """Generate a summary report of the human evaluation."""
    print("=" * 60)
    print("HUMAN EVALUATION REPORT")
    print("=" * 60)
    print(f"Items evaluated: {len(items)}")
    print(f"Dimensions: {', '.join(dim_avgs.keys())}")
    print(f"Average inter-annotator kappa: {kappa_avg:.4f}")
    print()

    # Score distribution
    all_scores = [info["overall"] for info in item_avgs.values()]
    if all_scores:
        print("Score Distribution:")
        bins = [0] * 5
        for s in all_scores:
            idx = min(int(s) - 1, 4)
            bins[max(0, idx)] += 1
        for i, count in enumerate(bins):
            bar = "#" * (count * 4)
            print(f"  Score {i+1}: {bar} ({count})")
        print()
        print(f"  Mean:   {sum(all_scores)/len(all_scores):.2f}")
        print(f"  Median: {sorted(all_scores)[len(all_scores)//2]:.2f}")
        print(f"  Min:    {min(all_scores):.2f}")
        print(f"  Max:    {max(all_scores):.2f}")

    # Best and worst items
    if item_avgs:
        sorted_items = sorted(item_avgs.items(), key=lambda x: x[1]["overall"])
        print()
        print("Lowest scored item:")
        worst_id, worst_info = sorted_items[0]
        worst_item = next(i for i in items if i["id"] == worst_id)
        print(f"  Q: {worst_item['question']}")
        print(f"  Score: {worst_info['overall']:.2f}")
        print()
        print("Highest scored item:")
        best_id, best_info = sorted_items[-1]
        best_item = next(i for i in items if i["id"] == best_id)
        print(f"  Q: {best_item['question']}")
        print(f"  Score: {best_info['overall']:.2f}")

    print()
    print("=" * 60)

generate_report(eval_items, item_avgs, dim_avgs, avg_kappa)

HUMAN EVALUATION REPORT
Items evaluated: 5
Dimensions: correctness, relevance, completeness, safety
Average inter-annotator kappa: 0.0000

Score Distribution:
  Score 1:  (0)
  Score 2:  (0)
  Score 3: #################### (5)
  Score 4:  (0)
  Score 5:  (0)

  Mean:   3.50
  Median: 3.58
  Min:    3.08
  Max:    3.67

Lowest scored item:
  Q: What did the Cheshire Cat say about madness?
  Score: 3.08

Highest scored item:
  Q: What did the Caterpillar ask Alice?
  Score: 3.67



In [27]:
# ## 11. Export Human Evaluation Results

In [28]:
output_dir = Path(r"D:\projects\python\MLCourse\03_agentic_ai\29_rag_evaluation")
output_dir.mkdir(exist_ok=True)

export = {
    "notebook": "03_human_evaluation",
    "num_items": len(eval_items),
    "graders": ["grader_A", "grader_B", "grader_C"],
    "dimensions": list(rubrics.keys()),
    "item_scores": {
        item_id: info for item_id, info in item_avgs.items()
    },
    "dimension_averages": dim_avgs,
    "inter_annotator_kappa": avg_kappa,
    "items": [
        {"id": item["id"], "question": item["question"], "answer": item["answer"][:200]}
        for item in eval_items
    ],
}

json_path = output_dir / "human_eval_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2)
print(f"[export] Saved human evaluation results to {json_path}")

[export] Saved human evaluation results to D:\projects\python\MLCourse\03_agentic_ai\29_rag_evaluation\human_eval_results.json


In [29]:
# ## Summary
#
# - Human evaluation captures quality dimensions that automated metrics miss
# - Multiple graders are essential: single-grader scores are unreliable
# - Inter-annotator agreement (Kappa) measures evaluation reliability
# - Aggregate scores across graders using averaging or majority vote
# - Define clear rubrics with explicit scale descriptions
# - Export results for tracking quality over time and across model versions